In [ ]:
!pip install ultralytics

from google.colab import drive
drive.mount('/content/drive')
print("✅ Ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.1 MB/s eta 0:00:00
Mounted at /content/drive
✅ Ready!


In [ ]:
import cv2
import joblib
import pandas as pd
import numpy as np
from ultralytics import YOLO
from collections import deque

# ---- Load model ----
rf_model = joblib.load("/content/drive/MyDrive/Badminton_RF_Model.pkl")

# Quick sanity check — confirm classes are strings, not integers
print("Model classes:", rf_model.classes_)

yolo = YOLO("yolov8n-pose.pt")

VIDEO_PATH  = "/content/drive/MyDrive/FRAMES OF 5/videos/video-2 .mp4"
OUTPUT_PATH = "/content/output_FINAL.mp4"

cap    = cv2.VideoCapture(VIDEO_PATH)
fps    = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"✅ Video loaded! FPS: {fps} | Size: {width}x{height} | Total Frames: {total}")

out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16)
]

COURT_CENTER_X = width / 2
COURT_CENTER_Y = height / 2

def draw_main_player(frame, kp_xy, box):
    x, y, w, h = box
    x1, y1 = int(x - w/2), int(y - h/2)
    x2, y2 = int(x + w/2), int(y + h/2)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    h_img, w_img = frame.shape[:2]
    kp_pixels = [(int(kx * w_img), int(ky * h_img)) for kx, ky in kp_xy]

    for px, py in kp_pixels:
        if px > 0 and py > 0:
            cv2.circle(frame, (px, py), 5, (0, 0, 255), -1)

    for i, j in SKELETON:
        if i < len(kp_pixels) and j < len(kp_pixels):
            p1, p2 = kp_pixels[i], kp_pixels[j]
            if p1[0] > 0 and p1[1] > 0 and p2[0] > 0 and p2[1] > 0:
                cv2.line(frame, p1, p2, (255, 255, 0), 2)
    return frame

def get_direction_label(dx_centroid, dy_centroid, thresh=2.0):
    if abs(dx_centroid) < thresh and abs(dy_centroid) < thresh:
        return "STILL"
    if abs(dy_centroid) > abs(dx_centroid):
        return "BACKWARD" if dy_centroid > 0 else "FORWARD"
    else:
        return "RIGHT" if dx_centroid > 0 else "LEFT"

history_len = 10
centroid_history = deque(maxlen=history_len)
speed_history = deque(maxlen=history_len)
court_start_x, court_start_y = None, None
cumulative_dist = 0.0

frame_num = 0
all_predictions = []

print("\nProcessing video... ⏳")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    label, confidence = "No Pose", 0.0

    results = yolo(frame, verbose=False)

    if (results[0].keypoints is not None and
        len(results[0].keypoints.xyn) > 0 and
        results[0].boxes is not None):
        try:
            boxes = results[0].boxes.xywh
            areas = boxes[:, 2] * boxes[:, 3]
            main_player = areas.argmax().item()

            kp  = results[0].keypoints.xyn[main_player].numpy()
            box = boxes[main_player].tolist()

            frame = draw_main_player(frame, kp, box)

            right_shoulder_x, right_shoulder_y = kp[6]
            left_shoulder_x,  left_shoulder_y  = kp[5]
            right_hip_x,      right_hip_y      = kp[12]
            left_hip_x,       left_hip_y       = kp[11]
            right_knee_x,     right_knee_y     = kp[14]
            left_knee_x,      left_knee_y      = kp[13]
            right_ankle_x,    right_ankle_y    = kp[16]
            left_ankle_x,     left_ankle_y     = kp[15]
            right_foot_x,     right_foot_y     = right_ankle_x, right_ankle_y
            left_foot_x,      left_foot_y      = left_ankle_x,  left_ankle_y

            hip_center_x = (right_hip_x + left_hip_x) / 2
            hip_center_y = (right_hip_y + left_hip_y) / 2
            stance_width = abs(right_ankle_x - left_ankle_x)
            dx = right_ankle_x - left_ankle_x
            dy = right_ankle_y - left_ankle_y

            centroid_x = np.mean([left_shoulder_x, right_shoulder_x, left_hip_x, right_hip_x])
            centroid_y = np.mean([left_shoulder_y, right_shoulder_y, left_hip_y, right_hip_y])

            if len(centroid_history) > 0:
                prev_x, prev_y = centroid_history[-1]
                frame_dist = np.sqrt((centroid_x - prev_x)**2 + (centroid_y - prev_y)**2)
                speed_val = frame_dist * fps
            else:
                frame_dist = 0.0
                speed_val = 0.0
                prev_x, prev_y = centroid_x, centroid_y

            cumulative_dist += frame_dist

            if court_start_x is None:
                court_start_x, court_start_y = centroid_x, centroid_y

            straight_distance = np.sqrt((centroid_x - court_start_x)**2 + (centroid_y - court_start_y)**2)
            path_length = cumulative_dist
            path_efficiency = straight_distance / path_length if path_length > 0 else 0.0

            centroid_history.append((centroid_x, centroid_y))
            speed_history.append(speed_val)

            rolling_speed_mean = np.mean(speed_history) if len(speed_history) > 0 else 0.0
            rolling_speed_std  = np.std(speed_history) if len(speed_history) > 1 else 0.0

            xs = [p[0] for p in centroid_history]
            ys = [p[1] for p in centroid_history]
            centroid_x_smooth = np.mean(xs)
            centroid_y_smooth = np.mean(ys)
            speed_smoothed = rolling_speed_mean

            acceleration = (speed_val - speed_history[-2]) if len(speed_history) >= 2 else 0.0

            movement_range_x = (max(xs) - min(xs)) if len(centroid_history) >= 2 else 0.0
            movement_range_y = (max(ys) - min(ys)) if len(centroid_history) >= 2 else 0.0

            dx_centroid = centroid_x - prev_x
            dy_centroid = centroid_y - prev_y
            direction_change = 1 if (frame_num > 1 and np.sign(dx_centroid) != np.sign(dx)) else 0
            trajectory_angle = np.degrees(np.arctan2(dy_centroid, dx_centroid)) if frame_dist > 0 else 0.0

            direction_label = get_direction_label(dx_centroid, dy_centroid)
            direction_code = {"STILL": 0, "FORWARD": 1, "BACKWARD": 2, "LEFT": 3, "RIGHT": 4}[direction_label]

            recovery_distance = np.sqrt((centroid_x - COURT_CENTER_X)**2 + (centroid_y - COURT_CENTER_Y)**2)
            recovery_time = recovery_distance / (speed_val + 1e-6)

            if centroid_y < height / 3:
                court_zone = 0
            elif centroid_y < 2 * height / 3:
                court_zone = 1
            else:
                court_zone = 2

            stability_index = 1.0 / (1.0 + rolling_speed_std)

            direction_onehot = {
                'direction_BACKWARD': 1 if direction_label == "BACKWARD" else 0,
                'direction_FORWARD':  1 if direction_label == "FORWARD" else 0,
                'direction_LEFT':     1 if direction_label == "LEFT" else 0,
                'direction_RIGHT':    1 if direction_label == "RIGHT" else 0,
                'direction_STILL':    1 if direction_label == "STILL" else 0,
            }

            feature_row = pd.DataFrame([{
                'right_knee_y': right_knee_y,
                'cumulative_distance': cumulative_dist,
                'straight_distance': straight_distance,
                'acceleration': acceleration,
                'right_ankle_y': right_ankle_y,
                'dx': dx,
                'left_hip_x': left_hip_x,
                'recovery_time': recovery_time,
                'right_hip_x': right_hip_x,
                'left_ankle_y': left_ankle_y,
                'movement_range_y': movement_range_y,
                'right_knee_x': right_knee_x,
                'left_hip_y': left_hip_y,
                'stance_width': stance_width,
                'rolling_speed_mean': rolling_speed_mean,
                'left_foot_y': left_foot_y,
                'centroid_y_smooth': centroid_y_smooth,
                'trajectory_angle': trajectory_angle,
                'right_foot_y': right_foot_y,
                'left_shoulder_y': left_shoulder_y,
                'left_shoulder_x': left_shoulder_x,
                'dy': dy,
                'centroid_x_smooth': centroid_x_smooth,
                'speed_smoothed': speed_smoothed,
                'right_foot_x': right_foot_x,
                'centroid_x': centroid_x,
                'right_ankle_x': right_ankle_x,
                'right_shoulder_x': right_shoulder_x,
                'right_shoulder_y': right_shoulder_y,
                'recovery_distance': recovery_distance,
                'path_efficiency': path_efficiency,
                'hip_center_x': hip_center_x,
                'left_ankle_x': left_ankle_x,
                'centroid_y': centroid_y,
                'direction_change': direction_change,
                'right_hip_y': right_hip_y,
                'rolling_speed_std': rolling_speed_std,
                'movement_range_x': movement_range_x,
                'distance': frame_dist,
                'hip_center_y': hip_center_y,
                'path_length': path_length,
                'left_knee_y': left_knee_y,
                'left_foot_x': left_foot_x,
                'left_knee_x': left_knee_x,
                'direction_code': direction_code,
                'speed': speed_val,
                'court_zone': court_zone,
                'stability_index': stability_index,
                **direction_onehot
            }])

            feature_row = feature_row[rf_model.feature_names_in_]

            pred = rf_model.predict(feature_row)
            confidence = rf_model.predict_proba(feature_row).max() * 100
            label = pred[0]   # rf_model.classes_ are already strings — no inverse_transform needed

        except Exception as e:
            label, confidence = "No Pose", 0.0
            print(f"Frame {frame_num} error: {e}")

    cv2.rectangle(frame, (0, 0), (600, 65), (0, 0, 0), -1)
    cv2.putText(frame, f"Footwork: {label}  {confidence:.1f}%", (10, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

    out.write(frame)
    all_predictions.append({'frame': frame_num, 'prediction': label, 'confidence': f"{confidence:.1f}%"})

    if frame_num % 100 == 0:
        print(f"  Processed {frame_num}/{total} frames...")

cap.release()
out.release()
print(f"\n✅ Done! Output saved to {OUTPUT_PATH}")

Model classes: ['Backhand_Backcourt' 'Backhand_Front' 'Backhand_Mid' 'Forehand_Backcourt' 'Forehand_Front' 'Forehand_Mid' 'Recovery_Ready']
✅ Video loaded! FPS: 25 | Size: 1920x1080 | Total Frames: 1063

Processing video... ⏳
  Processed 100/1063 frames...
  Processed 200/1063 frames...
  Processed 300/1063 frames...
  Processed 400/1063 frames...
  Processed 500/1063 frames...
  Processed 600/1063 frames...
  Processed 700/1063 frames...
  Processed 800/1063 frames...
  Processed 900/1063 frames...
  Processed 1000/1063 frames...

✅ Done! Output saved to /content/output_FINAL.mp4


In [ ]:
import joblib
rf_model = joblib.load("/content/drive/MyDrive/Badminton_RF_Model.pkl")
print(rf_model.feature_names_in_)
print(rf_model.n_features_in_)

['right_knee_y' 'cumulative_distance' 'straight_distance' 'acceleration'
 'right_ankle_y' 'dx' 'left_hip_x' 'recovery_time' 'right_hip_x'
 'left_ankle_y' 'movement_range_y' 'right_knee_x' 'left_hip_y'
 'stance_width' 'rolling_speed_mean' 'left_foot_y' 'centroid_y_smooth'
 'trajectory_angle' 'right_foot_y' 'left_shoulder_y' 'left_shoulder_x'
 'dy' 'centroid_x_smooth' 'speed_smoothed' 'right_foot_x' 'centroid_x'
 'right_ankle_x' 'right_shoulder_x' 'right_shoulder_y' 'recovery_distance'
 'path_efficiency' 'hip_center_x' 'left_ankle_x' 'centroid_y'
 'direction_change' 'right_hip_y' 'rolling_speed_std' 'movement_range_x'
 'distance' 'hip_center_y' 'path_length' 'left_knee_y' 'left_foot_x'
 'left_knee_x' 'direction_code' 'speed' 'court_zone' 'stability_index'
 'direction_BACKWARD' 'direction_FORWARD' 'direction_LEFT'
 'direction_RIGHT' 'direction_STILL']
53


In [ ]:
# Convert to dataframe
pred_df = pd.DataFrame(all_predictions)

print("--- Sample Predictions (first 10 frames) ---")
print(pred_df.head(10).to_string(index=False))

print("\n--- Footwork Distribution in Video ---")
print(pred_df['prediction'].value_counts())

--- Sample Predictions (first 10 frames) ---
 frame     prediction confidence
     1 Recovery_Ready      58.0%
     2 Recovery_Ready      60.0%
     3 Recovery_Ready      56.0%
     4 Recovery_Ready      56.0%
     5 Recovery_Ready      57.0%
     6 Recovery_Ready      55.0%
     7 Recovery_Ready      57.0%
     8 Recovery_Ready      57.0%
     9 Recovery_Ready      57.0%
    10 Recovery_Ready      59.0%

--- Footwork Distribution in Video ---
prediction
Backhand_Front        378
Recovery_Ready        254
Backhand_Mid          134
Forehand_Front        131
Forehand_Mid          128
Backhand_Backcourt     33
Forehand_Backcourt      5
Name: count, dtype: int64


In [ ]:
import shutil

src = "/content/output_FINAL.mp4"
dst = "/content/drive/MyDrive/output_FINAL.mp4"

shutil.copy(src, dst)
print("✅ Video copied to Google Drive!")
print("📁 Find it in: My Drive → video2_output.mp4")

✅ Video copied to Google Drive!
📁 Find it in: My Drive → video2_output.mp4
